In [ ]:
# LANGKAH 1 — Install dependency

%pip install -q --upgrade pymupdf requests

print("✅ Dependency berhasil di-install.")
print("PyMuPDF dan requests siap digunakan.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 59.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
✅ Dependency berhasil di-install.
PyMuPDF dan requests siap digunakan.


In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

OUTPUT_DIR = "/content/drive/MyDrive/BiLSTM-CRF/data/FathulQorib_OCR"
PAGES_DIR = os.path.join(OUTPUT_DIR, "pages")

os.makedirs(PAGES_DIR, exist_ok=True)

print("✅ Google Drive terhubung")
print("Folder output:", OUTPUT_DIR)

Mounted at /content/drive
✅ Google Drive terhubung
Folder output: /content/drive/MyDrive/FathulQorib_OCR


In [ ]:
# LANGKAH 3 — Ambil API Key dari Colab Secrets
# Membaca API key ZansLab dengan aman tanpa menampilkannya di output.

from google.colab import userdata

API_KEY = userdata.get("ZANSLAB_API_KEY")

if not API_KEY:
    raise ValueError("❌ ZANSLAB_API_KEY tidak ditemukan di Colab Secrets.")

print("✅ API key berhasil dibaca dari Colab Secrets.")

✅ API key berhasil dibaca dari Colab Secrets.


In [ ]:
# LANGKAH 4 — Upload PDF, simpan ke Google Drive, dan cek jumlah halaman
# PDF yang di-upload akan disimpan di folder FathulQorib_OCR agar aman saat runtime terputus.

from google.colab import files
import pymupdf
import os
import shutil

uploaded = files.upload()

if not uploaded:
    raise ValueError("❌ Tidak ada file yang di-upload.")

PDF_NAME = next(iter(uploaded.keys()))

if not PDF_NAME.lower().endswith(".pdf"):
    raise ValueError("❌ File yang dipilih bukan PDF.")

UPLOAD_PATH = f"/content/{PDF_NAME}"
PDF_PATH = os.path.join(OUTPUT_DIR, PDF_NAME)

# Simpan salinan PDF ke Google Drive
shutil.copy2(UPLOAD_PATH, PDF_PATH)

# Validasi PDF
try:
    doc = pymupdf.open(PDF_PATH)
except Exception as e:
    raise ValueError(f"❌ PDF gagal dibuka: {e}")

if doc.page_count == 0:
    raise ValueError("❌ PDF tidak memiliki halaman.")

print("✅ PDF berhasil dibuka")
print("Nama file     :", PDF_NAME)
print("Total halaman :", doc.page_count)
print("Lokasi        :", PDF_PATH)

Saving Kitab_Fathul_Qorib.pdf to Kitab_Fathul_Qorib.pdf
✅ PDF berhasil dibuka
Nama file     : Kitab_Fathul_Qorib.pdf
Total halaman : 368
Lokasi        : /content/drive/MyDrive/FathulQorib_OCR/Kitab_Fathul_Qorib.pdf


In [ ]:
# LANGKAH 5 — Atur halaman yang akan diproses
# Kita akan memproses halaman 19 sampai 350.
# Nomor halaman memakai nomor PDF yang dilihat manusia, mulai dari 1.

START_PAGE = 19
END_PAGE = 350

TOTAL_PAGES = doc.page_count

if START_PAGE < 1:
    raise ValueError("❌ START_PAGE minimal 1.")

if END_PAGE > TOTAL_PAGES:
    raise ValueError(f"❌ END_PAGE melebihi total halaman PDF ({TOTAL_PAGES}).")

if START_PAGE > END_PAGE:
    raise ValueError("❌ START_PAGE tidak boleh lebih besar dari END_PAGE.")

TOTAL_SELECTED = END_PAGE - START_PAGE + 1

print("✅ Rentang halaman valid")
print("Mulai halaman :", START_PAGE)
print("Sampai halaman:", END_PAGE)
print("Jumlah halaman:", TOTAL_SELECTED)

✅ Rentang halaman valid
Mulai halaman : 19
Sampai halaman: 350
Jumlah halaman: 332


In [ ]:
# LANGKAH 7 — Buat prompt OCR
# Prompt ini memerintahkan model melakukan transkripsi visual,
# mempertahankan harakat yang terlihat, tanpa memperbaiki berdasarkan bahasa Arab.

OCR_PROMPT = """
Anda adalah sistem TRANSKRIPSI OCR teks Arab, bukan editor bahasa Arab.

Tugas:
Ekstrak seluruh TEKS UTAMA KITAB yang benar-benar terlihat pada gambar.

WAJIB:
- Pertahankan semua harakat yang terlihat, termasuk fathah, kasrah, dhammah,
  sukun, syaddah, fathatain, kasratain, dhammatain, dan diakritik Arab lainnya.
- Transkripsikan sesuai dengan apa yang tercetak pada scan.
- Pertahankan urutan teks.
- Judul kitab, judul bab, fasal, subjudul, matan, syarah,
  dan isi asli Fathul Qorib harus tetap diambil.
- Satu paragraf logis ditulis sebagai satu paragraf.
- Pisahkan paragraf berbeda dengan satu baris kosong.

JANGAN:
- Jangan memperbaiki huruf atau harakat berdasarkan nahwu.
- Jangan memperbaiki berdasarkan sharaf.
- Jangan memperbaiki berdasarkan konteks.
- Jangan memperbaiki berdasarkan pengetahuan bahasa Arab.
- Jangan memperbaiki berdasarkan hafalan kitab.
- Jangan menebak teks yang tidak terlihat.
- Jangan menerjemahkan.
- Jangan meringkas.
- Jangan menambahkan penjelasan.
- Jangan hanya mengambil matan Taqrib.

BUANG:
- nomor halaman
- running header
- header berulang
- footer
- catatan kaki atau hawasy penerbit/tahqiq
- nomor atau simbol penanda footnote
- informasi penerbit/editor
- ornamen
- garis dekoratif
- teks marginal yang bukan isi utama kitab

Jika karakter benar-benar tidak dapat dibaca, gunakan:
[غير واضح]

OUTPUT:
HANYA teks Arab hasil transkripsi.

Jangan menulis "Hasil OCR:".
Jangan gunakan Markdown.
Jangan gunakan triple backticks.
Jangan memberikan komentar atau penjelasan apa pun.
"""

print("✅ OCR_PROMPT siap digunakan.")

✅ OCR_PROMPT siap digunakan.


In [ ]:
# langkah 8 — Konfigurasi API ZansLab
# Digunakan oleh fungsi process_page() untuk menentukan endpoint dan model.

API_URL = "https://zanslab.id/v1/chat/completions"
MODEL = "op/gpt-5.6-sol"

print("✅ Konfigurasi API siap")
print("Model:", MODEL)

✅ Konfigurasi API siap
Model: op/gpt-5.6-sol


In [ ]:
# LANGKAH 10 — Fungsi OCR lengkap: render, retry, validasi, checkpoint, dan skip
# Fungsi ini akan dipakai untuk memproses halaman 19–350  satu per satu.
# Jika TXT halaman sudah ada dan tidak kosong, API tidak dipanggil lagi.

import os
import base64
import time
import requests
import re

RETRY_STATUS = {500, 502, 503, 504}
MAX_ATTEMPTS = 4
RETRY_DELAYS = [5, 10, 20]


def clean_ocr_output(text):
    # Bersihkan hanya wrapper Markdown, tanpa mengubah teks/harakat Arab.
    if not text:
        return ""

    text = text.strip()
    text = re.sub(r"^```(?:arabic)?\s*", "", text, flags=re.IGNORECASE)
    text = re.sub(r"\s*```$", "", text)

    return text.strip()


def process_page(page_number):

    output_path = os.path.join(
        PAGES_DIR,
        f"page_{page_number:03d}.txt"
    )

    # CHECKPOINT:
    # Jika file sudah ada dan berisi teks, halaman langsung dilewati.
    if os.path.exists(output_path) and os.path.getsize(output_path) > 0:
        print(f"⏭️ Halaman {page_number}: SKIP")

        return {
            "page": page_number,
            "status": "skipped",
            "attempts": 0,
            "prompt_tokens": 0,
            "completion_tokens": 0,
            "total_tokens": 0,
            "error": ""
        }

    # Render halaman PDF 3x
    page_index = page_number - 1
    page = doc.load_page(page_index)

    matrix = pymupdf.Matrix(3, 3)
    pix = page.get_pixmap(matrix=matrix, alpha=False)

    temp_image = f"/content/page_{page_number:03d}.jpg"
    pix.save(temp_image, jpg_quality=95)

    # Encode gambar ke base64
    with open(temp_image, "rb") as f:
        image_base64 = base64.b64encode(f.read()).decode("utf-8")

    payload = {
        "model": MODEL,
        "messages": [
            {
                "role": "user",
                "content": [
                    {
                        "type": "text",
                        "text": OCR_PROMPT
                    },
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": f"data:image/jpeg;base64,{image_base64}"
                        }
                    }
                ]
            }
        ],
        "temperature": 0,
        "max_tokens": 8000
    }

    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type": "application/json"
    }

    last_error = ""

    for attempt in range(1, MAX_ATTEMPTS + 1):

        print(
            f"🔄 Halaman {page_number} — "
            f"percobaan {attempt}/{MAX_ATTEMPTS}"
        )

        try:
            response = requests.post(
                API_URL,
                headers=headers,
                json=payload,
                timeout=180
            )

            # Jika berhasil
            if response.status_code == 200:

                data = response.json()

                raw_text = data["choices"][0]["message"]["content"]
                text = clean_ocr_output(raw_text)

                if not text:
                    last_error = "Output OCR kosong"
                    print("⚠️ Output OCR kosong.")

                    if attempt < MAX_ATTEMPTS:
                        time.sleep(RETRY_DELAYS[attempt - 1])
                        continue
                    break

                usage = data.get("usage") or {}

                # Simpan checkpoint langsung
                with open(output_path, "w", encoding="utf-8") as f:
                    f.write(text)

                # Hapus gambar sementara
                if os.path.exists(temp_image):
                    os.remove(temp_image)

                print(f"✅ Halaman {page_number}: SUCCESS")

                return {
                    "page": page_number,
                    "status": "success",
                    "attempts": attempt,
                    "prompt_tokens": usage.get("prompt_tokens", 0) or 0,
                    "completion_tokens": usage.get("completion_tokens", 0) or 0,
                    "total_tokens": usage.get("total_tokens", 0) or 0,
                    "error": ""
                }

            error_text = response.text
            last_error = f"HTTP {response.status_code}: {error_text}"

            # Error permanen → jangan retry
            if (
                response.status_code in {401, 402, 403}
                or "INVALID_API_KEY" in error_text.upper()
                or "INSUFFICIENT_BALANCE" in error_text.upper()
            ):
                print("❌ Error API permanen:")
                print(error_text)

                if os.path.exists(temp_image):
                    os.remove(temp_image)

                raise RuntimeError(last_error)

            # Error server sementara → retry
            if response.status_code in RETRY_STATUS:

                print(f"⚠️ HTTP {response.status_code}")

                if attempt < MAX_ATTEMPTS:
                    delay = RETRY_DELAYS[attempt - 1]
                    print(f"🔁 Retry dalam {delay} detik...")
                    time.sleep(delay)
                    continue

            # Error lain
            break

        except (requests.exceptions.Timeout,
                requests.exceptions.ConnectionError) as e:

            last_error = str(e)
            print("⚠️ Network/timeout:", e)

            if attempt < MAX_ATTEMPTS:
                delay = RETRY_DELAYS[attempt - 1]
                print(f"🔁 Retry dalam {delay} detik...")
                time.sleep(delay)
                continue

    # Jika semua percobaan gagal
    if os.path.exists(temp_image):
        os.remove(temp_image)

    print(f"❌ Halaman {page_number}: FAILED")

    return {
        "page": page_number,
        "status": "failed",
        "attempts": MAX_ATTEMPTS,
        "prompt_tokens": 0,
        "completion_tokens": 0,
        "total_tokens": 0,
        "error": last_error
    }


print("✅ Fungsi process_page() siap digunakan.")

✅ Fungsi process_page() siap digunakan.


In [ ]:
 # LANGKAH 11 — Proses halaman 19–350 dan simpan progress.csv
# Setiap halaman langsung dicatat ke Google Drive.
# Jika Colab terputus, halaman TXT yang sudah berhasil akan di-SKIP saat dijalankan lagi.

import csv
import os

PROGRESS_PATH = os.path.join(OUTPUT_DIR, "progress.csv")

PROGRESS_FIELDS = [
    "page",
    "status",
    "attempts",
    "prompt_tokens",
    "completion_tokens",
    "total_tokens",
    "error"
]

# Membaca progress lama jika sebelumnya sudah pernah dijalankan.
progress_data = {}

if os.path.exists(PROGRESS_PATH):
    with open(PROGRESS_PATH, "r", encoding="utf-8-sig", newline="") as f:
        reader = csv.DictReader(f)

        for row in reader:
            try:
                progress_data[int(row["page"])] = row
            except (ValueError, KeyError):
                pass


def save_progress():
    # Simpan ulang progress berdasarkan urutan nomor halaman.
    with open(PROGRESS_PATH, "w", encoding="utf-8-sig", newline="") as f:
        writer = csv.DictWriter(
            f,
            fieldnames=PROGRESS_FIELDS,
            quoting=csv.QUOTE_MINIMAL
        )

        writer.writeheader()

        for page_num in sorted(progress_data):
            writer.writerow(progress_data[page_num])


print(f"🚀 Memproses halaman {START_PAGE}–{END_PAGE}")
print("=" * 60)

for page_number in range(START_PAGE, END_PAGE + 1):

    try:
        result = process_page(page_number)

        # Jika halaman hanya di-SKIP dan sebelumnya punya catatan sukses,
        # pertahankan token usage lama agar statistik tidak hilang.
        old = progress_data.get(page_number)

        if (
            result["status"] == "skipped"
            and old
            and old.get("status") == "success"
        ):
            result["prompt_tokens"] = old.get("prompt_tokens", 0)
            result["completion_tokens"] = old.get("completion_tokens", 0)
            result["total_tokens"] = old.get("total_tokens", 0)

        progress_data[page_number] = result

        # WAJIB simpan setelah setiap halaman.
        save_progress()

        print(f"💾 Progress halaman {page_number} tersimpan.")
        print("-" * 60)

    except RuntimeError as e:
        # Error permanen seperti API key/balance → hentikan proses.
        progress_data[page_number] = {
            "page": page_number,
            "status": "failed",
            "attempts": 1,
            "prompt_tokens": 0,
            "completion_tokens": 0,
            "total_tokens": 0,
            "error": str(e)
        }

        save_progress()

        print("\n🛑 Proses dihentikan karena error API permanen.")
        print(e)
        break

print("\n✅ Proses selesai / dihentikan dengan aman.")
print("Progress:", PROGRESS_PATH)

🚀 Memproses halaman 19–350
⏭️ Halaman 19: SKIP
💾 Progress halaman 19 tersimpan.
------------------------------------------------------------
⏭️ Halaman 20: SKIP
💾 Progress halaman 20 tersimpan.
------------------------------------------------------------
⏭️ Halaman 21: SKIP
💾 Progress halaman 21 tersimpan.
------------------------------------------------------------
⏭️ Halaman 22: SKIP
💾 Progress halaman 22 tersimpan.
------------------------------------------------------------
⏭️ Halaman 23: SKIP
💾 Progress halaman 23 tersimpan.
------------------------------------------------------------
⏭️ Halaman 24: SKIP
💾 Progress halaman 24 tersimpan.
------------------------------------------------------------
⏭️ Halaman 25: SKIP
💾 Progress halaman 25 tersimpan.
------------------------------------------------------------
⏭️ Halaman 26: SKIP
💾 Progress halaman 26 tersimpan.
------------------------------------------------------------
⏭️ Halaman 27: SKIP
💾 Progress halaman 27 tersimpan.
--------

In [ ]:
# DATASET FINAL — Buat ulang dataset.csv
# Halaman peta/non-isi dikeluarkan dari dataset.
# Halaman 135 tetap digunakan karena merupakan teks kitab.

import csv
import os

EXCLUDED_PAGES = {146, 150, 151}

DATASET_PATH = os.path.join(OUTPUT_DIR, "dataset.csv")
PROGRESS_PATH = os.path.join(OUTPUT_DIR, "progress.csv")

# Baca progress.csv kembali agar token usage tetap tersedia
progress_data = {}

if os.path.exists(PROGRESS_PATH):
    with open(PROGRESS_PATH, "r", encoding="utf-8-sig", newline="") as f:
        reader = csv.DictReader(f)

        for row in reader:
            try:
                progress_data[int(row["page"])] = row
            except (ValueError, KeyError):
                pass


DATASET_FIELDS = [
    "page",
    "text_diacritized",
    "status",
    "prompt_tokens",
    "completion_tokens",
    "total_tokens",
    "paragraph_count"
]

rows = []
missing_pages = []

for page_number in range(START_PAGE, END_PAGE + 1):

    # Jangan masukkan halaman peta/non-isi
    if page_number in EXCLUDED_PAGES:
        continue

    page_path = os.path.join(
        PAGES_DIR,
        f"page_{page_number:03d}.txt"
    )

    # Validasi file
    if not os.path.exists(page_path) or os.path.getsize(page_path) == 0:
        missing_pages.append(page_number)
        continue

    with open(page_path, "r", encoding="utf-8") as f:
        text = f.read().strip()

    if not text:
        missing_pages.append(page_number)
        continue

    progress = progress_data.get(page_number, {})

    paragraphs = [
        p.strip()
        for p in text.split("\n\n")
        if p.strip()
    ]

    rows.append({
        "page": page_number,
        "text_diacritized": text,
        "status": "success",
        "prompt_tokens": progress.get("prompt_tokens", 0) or 0,
        "completion_tokens": progress.get("completion_tokens", 0) or 0,
        "total_tokens": progress.get("total_tokens", 0) or 0,
        "paragraph_count": len(paragraphs)
    })


# Jangan membuat dataset diam-diam jika ada halaman yang seharusnya ada tetapi hilang
if missing_pages:
    raise ValueError(
        f"❌ Ada halaman valid yang hilang/kosong: {missing_pages}"
    )


with open(DATASET_PATH, "w", encoding="utf-8-sig", newline="") as f:
    writer = csv.DictWriter(
        f,
        fieldnames=DATASET_FIELDS,
        quoting=csv.QUOTE_ALL
    )

    writer.writeheader()
    writer.writerows(rows)


print("✅ dataset.csv FINAL berhasil dibuat")
print("Rentang sumber   :", f"{START_PAGE}–{END_PAGE}")
print("Halaman excluded :", sorted(EXCLUDED_PAGES))
print("Jumlah excluded  :", len(EXCLUDED_PAGES))
print("Jumlah data      :", len(rows))
print("Lokasi           :", DATASET_PATH)

✅ dataset.csv FINAL berhasil dibuat
Rentang sumber   : 19–350
Halaman excluded : [146, 150, 151]
Jumlah excluded  : 3
Jumlah data      : 329
Lokasi           : /content/drive/MyDrive/FathulQorib_OCR/dataset.csv


In [ ]:
# FULL TEXT FINAL — Buat ulang full_text.txt
# Menggabungkan hanya halaman isi kitab.
# Halaman peta/non-isi tidak dimasukkan.

FULL_TEXT_PATH = os.path.join(OUTPUT_DIR, "full_text.txt")

all_pages_text = []

for page_number in range(START_PAGE, END_PAGE + 1):

    if page_number in EXCLUDED_PAGES:
        continue

    page_path = os.path.join(
        PAGES_DIR,
        f"page_{page_number:03d}.txt"
    )

    if not os.path.exists(page_path) or os.path.getsize(page_path) == 0:
        raise FileNotFoundError(
            f"❌ File halaman {page_number} tidak ditemukan/kosong."
        )

    with open(page_path, "r", encoding="utf-8") as f:
        text = f.read().strip()

    if not text:
        raise ValueError(
            f"❌ Teks halaman {page_number} kosong."
        )

    all_pages_text.append(text)


full_text = "\n\n".join(all_pages_text)

with open(FULL_TEXT_PATH, "w", encoding="utf-8") as f:
    f.write(full_text)


print("✅ full_text.txt FINAL berhasil dibuat")
print("Halaman tergabung:", len(all_pages_text))
print("Halaman excluded :", sorted(EXCLUDED_PAGES))
print("Jumlah karakter :", len(full_text))
print("Lokasi          :", FULL_TEXT_PATH)

✅ full_text.txt FINAL berhasil dibuat
Halaman tergabung: 329
Halaman excluded : [146, 150, 151]
Jumlah karakter : 452325
Lokasi          : /content/drive/MyDrive/FathulQorib_OCR/full_text.txt


In [ ]:
# LANGKAH 14 — Buat errors.csv dan statistik akhir FINAL
# Halaman dianggap SUCCESS jika file TXT tersedia dan tidak kosong.
# Token tetap diambil dari progress.csv.

import csv
import os

ERRORS_PATH = os.path.join(OUTPUT_DIR, "errors.csv")

success_count = 0
failed_count = 0

total_prompt_tokens = 0
total_completion_tokens = 0
total_tokens = 0

errors = []

for page_number in range(START_PAGE, END_PAGE + 1):

    page_path = os.path.join(
        PAGES_DIR,
        f"page_{page_number:03d}.txt"
    )

    progress = progress_data.get(page_number, {})

    # Cek hasil OCR sebenarnya
    if os.path.exists(page_path) and os.path.getsize(page_path) > 0:

        with open(page_path, "r", encoding="utf-8") as f:
            text = f.read().strip()

        if text:
            success_count += 1
        else:
            failed_count += 1
            errors.append({
                "page": page_number,
                "error": "File TXT kosong"
            })

    else:
        failed_count += 1
        errors.append({
            "page": page_number,
            "error": progress.get("error", "File TXT tidak ditemukan")
        })

    # Token usage
    total_prompt_tokens += int(progress.get("prompt_tokens", 0) or 0)
    total_completion_tokens += int(progress.get("completion_tokens", 0) or 0)
    total_tokens += int(progress.get("total_tokens", 0) or 0)


# Simpan errors.csv
with open(ERRORS_PATH, "w", encoding="utf-8-sig", newline="") as f:
    writer = csv.DictWriter(
        f,
        fieldnames=["page", "error"],
        quoting=csv.QUOTE_ALL
    )

    writer.writeheader()
    writer.writerows(errors)


print("✅ STATISTIK FINAL")
print("Halaman target      :", TOTAL_SELECTED)
print("Success             :", success_count)
print("Failed              :", failed_count)

print("\n--- TOKEN USAGE ---")
print("Prompt tokens       :", total_prompt_tokens)
print("Completion tokens   :", total_completion_tokens)
print("Total tokens        :", total_tokens)

print("\nErrors.csv           :", ERRORS_PATH)

✅ STATISTIK FINAL
Halaman target      : 332
Success             : 332
Failed              : 0

--- TOKEN USAGE ---
Prompt tokens       : 2057172
Completion tokens   : 415452
Total tokens        : 2472624

Errors.csv           : /content/drive/MyDrive/FathulQorib_OCR/errors.csv


In [ ]:
# LANGKAH 15 — Validasi final seluruh halaman
# Mengecek apakah semua file TXT ada, tidak kosong, dan bisa dibaca.

missing_pages = []
empty_pages = []
valid_pages = []

for page_number in range(START_PAGE, END_PAGE + 1):

    page_path = os.path.join(
        PAGES_DIR,
        f"page_{page_number:03d}.txt"
    )

    if not os.path.exists(page_path):
        missing_pages.append(page_number)
        continue

    if os.path.getsize(page_path) == 0:
        empty_pages.append(page_number)
        continue

    with open(page_path, "r", encoding="utf-8") as f:
        text = f.read().strip()

    if not text:
        empty_pages.append(page_number)
    else:
        valid_pages.append(page_number)


print("✅ VALIDASI FINAL")
print("Rentang target :", f"{START_PAGE}–{END_PAGE}")
print("Halaman target :", TOTAL_SELECTED)
print("File valid     :", len(valid_pages))
print("File hilang    :", len(missing_pages))
print("File kosong    :", len(empty_pages))

if missing_pages:
    print("Halaman hilang :", missing_pages)

if empty_pages:
    print("Halaman kosong :", empty_pages)

if len(valid_pages) == TOTAL_SELECTED:
    print(
        f"\n🎉 Semua halaman {START_PAGE}–{END_PAGE} "
        "tersedia dan tidak kosong."
    )
else:
    print("\n⚠️ Ada halaman yang perlu diperiksa.")

✅ VALIDASI FINAL
Rentang target : 19–350
Halaman target : 332
File valid     : 332
File hilang    : 0
File kosong    : 0

🎉 Semua halaman 19–350 tersedia dan tidak kosong.
